In [3]:
!apt update -q && apt-get install -qy openjdk-21-jdk-headless > /dev/null 2>&1

import subprocess
from IPython.core.magic import register_cell_magic

class PersistentJShell:
    def __init__(self):
        self.proc = subprocess.Popen(
            ["jshell", "-s"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

    def execute(self, code):
        if self.proc.poll() is not None:
            raise RuntimeError("JShell process has terminated.")

        marker = "___JSHELL_CELL_END___"
        payload = code.strip() + f'\nSystem.out.println("{marker}");\n'
        self.proc.stdin.write(payload)
        self.proc.stdin.flush()

        lines = []
        while True:
            line = self.proc.stdout.readline()
            if not line:
                break
            if marker in line:
                break
            lines.append(line)

        return "".join(lines)

_jshell_instance = PersistentJShell()

@register_cell_magic
def jshell(line, cell):
    out = _jshell_instance.execute(cell)
    if out:
        print(out, end="")


Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:5 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:7 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Fetched 3,917 B in 1s (4,842 B/s)
Reading package lists...
Building dependency tree...
Reading state information...
37 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [4]:
%%jshell

void runPy(String code) throws Exception {
    String marker = "___END_OF_PYTHON_CELL___";

    // Base64 encode code payload to bypass all quote & newline syntax errors
    String b64 = java.util.Base64.getEncoder().encodeToString(code.getBytes(java.nio.charset.StandardCharsets.UTF_8));

    // Guarantee marker printing using try...finally and display trailing expressions using ast
    String payload = String.format(
        "import base64, ast, traceback\n" +
        "try:\n" +
        "    code_str = base64.b64decode('%s').decode('utf-8')\n" +
        "    tree = ast.parse(code_str)\n" +
        "    if tree.body and isinstance(tree.body[-1], ast.Expr):\n" +
        "        exec(compile(ast.Module(body=tree.body[:-1], type_ignores=[]), '<string>', 'exec'), globals())\n" +
        "        res = eval(compile(ast.Expression(body=tree.body[-1].value), '<string>', 'eval'), globals())\n" +
        "        if res is not None:\n" +
        "            print(res)\n" +
        "    else:\n" +
        "        exec(compile(tree, '<string>', 'exec'), globals())\n" +
        "except Exception:\n" +
        "    traceback.print_exc()\n" +
        "finally:\n" +
        "    print('%s', flush=True)\n",
        b64, marker
    );

    pyIn.println(payload);

    String line;
    while ((line = pyOut.readLine()) != null) {
        if (line.contains(marker)) break;
        System.out.println(line);
    }
}

import java.io.BufferedReader;
import java.io.InputStreamReader;
import java.io.PrintStream;
import java.io.IOException;

Process pythonProcess;
PrintStream pyIn;
BufferedReader pyOut;

try {
    ProcessBuilder pb = new ProcessBuilder("python3", "-i", "-u");
    // Redirect stderr to stdout to prevent pipe deadlock and capture Python errors
    pb.redirectErrorStream(true);
    pythonProcess = pb.start();

    pyIn = new PrintStream(pythonProcess.getOutputStream(), true);
    pyOut = new BufferedReader(new InputStreamReader(pythonProcess.getInputStream()));

    System.out.println("Python interpreter initialized safely.");
} catch (IOException e) {
    e.printStackTrace();
}

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
->   ->  ->  ->  ->  ->  ->  ->  ->  ->  ->  ->  Python interpreter initialized safely.


In [5]:
%%jshell

runPy("""
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path ; from IPython.display import display, HTML
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():os.system('git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming');

NOTEBOOK_DIR = REPO_ROOT / 'assignments/assignment_6'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')
""");


->  Python 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0] on linux
Type "help", "copyright", "credits" or "license" for more information.
>>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Working directory: /content/BITS_programming/assignments/assignment_6


In [6]:
%%jshell

runPy("""
# Data Reduction - Attribute Subset Selection
#
# In this lab, we'll be learning about different techniques that are generally
# used to select the best subset of attributes for a given model/algorith
#
# Attribute selection is the process of selection of attributes for the purpose
# of your analysis.
#
# Different variables can have different impact at the output of your analysis
# or impact the performance of your model/algorithm overall, which makes it
# important to choose the right attributes.
#
# There are multiple methods through which we can choose the right attributes:
#
# a. Forward Selection
# b. Backward Selection
# c. Decision Tree Induction
#
# We have primarily done this for Linear Regression and Random Forest model
""");


In [7]:
%%jshell

runPy("""
# Download the dataset file to local storage to current directory (from publicly
# shared drive file)
#
# Original source:
# https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques
os.system('gdown 1Yhw8tBCmjYrDZ5Rm1xPrP2y5bmBmxoEP');
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Downloading...
From: https://drive.google.com/uc?id=1Yhw8tBCmjYrDZ5Rm1xPrP2y5bmBmxoEP
To: /content/BITS_programming/assignments/assignment_6/housing.csv

  0%|                                               | 0.00/461k [00:00<?, ?B/s]
100%|███████████████████████████████████████| 461k/461k [00:00<00:00, 97.8MB/s]
0


In [8]:
%%jshell

runPy("""
# Import required libraries
import pandas as pd

from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression
""");


In [9]:
%%jshell

runPy("""
# Load the dataset file into memory
DATASET_PATH = r'https://raw.githubusercontent.com/2025em1100422-ui/kaggledatset/564eb719455f768c2a008bc893dab1f6eb1e8b98/train.csv'
df = pd.read_csv(DATASET_PATH)
df
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ...         Id  MSSubClass MSZoning  ...  SaleType  SaleCondition SalePrice
0        1          60       RL  ...        WD         Normal    208500
1        2          20       RL  ...        WD         Normal    181500
2        3          60       RL  ...        WD         Normal    223500
3        4          70       RL  ...        WD        Abnorml    140000
4        5          60       RL  ...        WD         Normal    250000
...    ...         ...      ...  ...       ...            ...       ...
1455  1456          60       RL  ...        WD         Normal    175000
1456  1457          20       RL  ...        WD         Normal    210000
1457  1458          70       RL  ...        WD         Normal    266500
1458  1459          20       RL  ...        WD         Normal    142125
1459  1460          20       RL  ...        WD         Normal    147500

[1460 rows x 81 columns]


In [10]:
%%jshell

runPy("""
# Tuple Reduction: Reduces the number of data instances using techniques such as
# sampling and clustering

# Dimensionality Reduction will be covered in the next lab 6.2

# Data Cube Aggregation:
# Summarizes data by aggregating values at different levels (e.g., daily to
# monthly sales).

# Data Compression:
# Uses encoding methods (e.g., wavelet transforms, run-length encoding) to
# reduce data size.
""");


In [11]:
%%jshell

runPy("""
# We'll be considering a scenario where we want to figure out the best
# attributes to predict the SalePrice of a house
""");


In [12]:
%%jshell

runPy("""
model = LinearRegression()

target_variable = "SalePrice"

# Pre-processing the dataset
df_processed = df.fillna(0)

X = df_processed.drop(target_variable, axis=1).select_dtypes(include='number')
Y = df_processed[target_variable]

model.fit(X, Y)
import numpy as np
import matplotlib.pyplot as plt

# Get predictions from the linear model
Y_pred = model.predict(X)

# Create a scatter plot of actual vs predicted
plt.figure(figsize=(8, 6))
plt.scatter(Y, Y_pred, alpha=0.6, color='blue')
plt.plot([Y.min(), Y.max()], [Y.min(), Y.max()], 'k--', lw=2)  # Line y=x
plt.xlabel('Actual SalePrice')
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Text(0.5, 0, 'Actual SalePrice')


# New Section

In [13]:
%%jshell

runPy("""
# 1. Forward Selection
#
# In forward selection, we start with a minimal set and then include more
# attributes one by one as we find them positively impactful for the analysis
# being performed.
#
# We'll be sklearn's SequentialFeatureSelector to achieve the same
""");


In [14]:
%%jshell

runPy("""
sfs = SequentialFeatureSelector(model, n_features_to_select=5, direction='forward', cv=5, n_jobs=-1)
sfs.fit(X, Y)

# Get the names of the selected features
selected_indices_forward = sfs.get_support(indices=True)
selected_features_wrapper = df.columns[selected_indices_forward]

print("\\nSelected features (Wrapper Method - Forward Elimination):")
print(selected_features_wrapper)

# The methods selected for
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... 
Selected features (Wrapper Method - Forward Elimination):
Index(['MSSubClass', 'LotArea', 'Utilities', 'HouseStyle', 'MasVnrArea'], dtype='object')


In [15]:
%%jshell

runPy("""
# 2. Backward Selection
#
# In backward selection, we start the complete set and exclude one attribute
# at a time, until we have the desired/right number of attributes.
""");


In [16]:
%%jshell

runPy("""
# We'll be using SequentialFeatureSelector to achieve this with a backward
# direction
#
# n_features_to_select determines the number of features that the model selects
#
# direction='backward' mentioned that we would be considering all the features
# first and then removing them as per their importance
#
# cv parameter refers to the total number of folds for cross-validation
#
# n_jobs refers to the number of jobs to be used and "n_jobs=-1" specifically
# implies that the maximum number of jobs can be used
sfs = SequentialFeatureSelector(model, n_features_to_select=5,
                                direction='backward', cv=5, n_jobs=-1)
sfs.fit(X, Y)

# Get the names of the selected features
selected_indices_backward = sfs.get_support(indices=True)
selected_features_wrapper = df.columns[selected_indices_backward]

print("\\nSelected features (Wrapper Method - Backward Elimination):")
print(selected_features_wrapper)

# All the features were first considered and then based on their feature
# importance calculated by the model were
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... 
Selected features (Wrapper Method - Backward Elimination):
Index(['MSSubClass', 'LotArea', 'Alley', 'HouseStyle', 'OverallQual'], dtype='object')


In [17]:
%%jshell

runPy("""
# 3. Decision Tree Induction
#
# Decision Trees can be used for feature selection by examining the importance
# of each feature in the tree building process. Features that appear higher in
# the tree or are used more frequently in splitting decisions are considered
# more important.
#
# We can access feature importance scores from a trained Decision Tree model
# to rank and select features.
""");


In [18]:
%%jshell

runPy("""
from sklearn.tree import DecisionTreeRegressor
import matplotlib.pyplot as plt

# Initialize and train a Decision Tree Regressor model
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X, Y)

# Get feature importances
feature_importances = dt_model.feature_importances_

# Create a pandas Series for easier visualization
importance_series = pd.Series(feature_importances, index=X.columns)

# Sort feature importances and display the top ones
top_features = importance_series.sort_values(ascending=False)

print("\\nFeature importances from Decision Tree:")
display(top_features)

#  Visualize feature importances
plt.figure(figsize=(12, 6))
top_features.head(10).plot(kind='bar')
plt.title('Top 10 Feature Importances (Decision Tree)')
plt.ylabel('Importance')
plt.xlabel('Features')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/content/regression_plot.png')
# Get the feature sets from each method
forward_features = df.columns[selected_indices_forward]
backward_features = df.columns[selected_indices_backward]
tree_features = top_features.head(5).index # Changed to take top 5 features

# Create a DataFrame to display them
feature_table = pd.DataFrame({
    'Forward Selection': forward_features,
    'Backward Selection': backward_features,
    'Decision Tree Importance': tree_features
})

print("Selected Features by All Methods:")
display(feature_table)
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... 
Feature importances from Decision Tree:
OverallQual      0.640979
GrLivArea        0.101922
2ndFlrSF         0.041758
TotalBsmtSF      0.033788
1stFlrSF         0.025740
GarageCars       0.023287
BsmtFinSF1       0.022791
YearBuilt        0.018737
GarageArea       0.013240
GarageYrBlt      0.010260
LotArea          0.009693
LotFrontage      0.007054
YearRemodAdd     0.006937
KitchenAbvGr     0.005804
BsmtUnfSF        0.005714
MoSold           0.005644
TotRmsAbvGrd     0.004910
OverallCond      0.003890
Id               0.003041
BedroomAbvGr     0.002950
OpenPorchSF      0.002621
ScreenPorch      0.001780
EnclosedPorch    0.001484
YrSold           0.001144
WoodDeckSF       0.001098
Fireplaces       0.000783
BsmtFullBath     0.000753
MasVnrArea       0.000603
MSSubClass       0.000489
FullBath         0.000483
BsmtFinSF2       0.000261
HalfBath         0.000172
BsmtHalfBath     0.000076
PoolArea         0.000061
MiscVal

In [19]:
%%jshell

runPy("""
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))
""");


->  >>> >>> ... ... ... ... ... ... ... ... ... ... ... ... ... ... Current Time in IST: 2026-09-28 20:27:32
